# Change Detection with Repeat LiDAR Surveys

Repeat LiDAR surveys quantify terrain change: landslides, erosion,
construction, flood deposition, mine subsidence, or glacier retreat.

Workflow:
1. Align two epoch clouds with ICP (removes systematic offset)
2. Compute point-wise M3C2-style signed distances
3. Threshold to separate signal from noise
4. Map and quantify change volumes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from occulus.types import PointCloud
from occulus.normals import estimate_normals
from occulus.registration import icp_point_to_plane
from occulus.filters import voxel_downsample

## Simulate Pre/Post Survey (Landslide Scenario)

In [ ]:
rng = np.random.default_rng(42)
n = 30_000

# Hillside terrain — gentle slope
x = rng.uniform(0, 100, n)
y = rng.uniform(0, 100, n)
z_pre = 10 + 0.2 * x + 0.05 * y + 2 * np.sin(x / 20) + rng.normal(0, 0.05, n)

# Post: apply a simulated landslide displacement in a triangular zone
z_post = z_pre.copy()
# Depletion zone (material removed): x 20-50, y 30-70
depletion = (x > 20) & (x < 50) & (y > 30) & (y < 70)
z_post[depletion] -= 2.5 * (1 - np.abs(x[depletion] - 35) / 15) * rng.uniform(0.6, 1.0, depletion.sum())

# Deposition zone (material added): x 50-80, y 35-65
deposition = (x > 50) & (x < 80) & (y > 35) & (y < 65)
z_post[deposition] += 1.8 * (1 - np.abs(x[deposition] - 65) / 15) * rng.uniform(0.5, 1.0, deposition.sum())

# Add sensor noise
z_post += rng.normal(0, 0.04, n)

pre = PointCloud(np.column_stack([x, y, z_pre]).astype(np.float64))
post = PointCloud(np.column_stack([x, y, z_post]).astype(np.float64))

print(f'Pre-survey : {pre.n_points:,} points')
print(f'Post-survey: {post.n_points:,} points')

## ICP Alignment (Remove Systematic Offset)

In [ ]:
# Add a small artificial misalignment to simulate GPS drift
drift = np.array([0.08, -0.05, 0.02])
post_drifted = PointCloud((post.xyz + drift).astype(np.float64))

# Align with ICP
pre_ds = voxel_downsample(pre, 0.5)
post_ds = voxel_downsample(post_drifted, 0.5)
pre_n = estimate_normals(pre_ds, radius=2.0)
post_n = estimate_normals(post_ds, radius=2.0)

result = icp_point_to_plane(post_n, pre_n, max_correspondence_distance=0.3, max_iterations=50)
print(f'ICP: fitness={result.fitness:.4f}, RMSE={result.inlier_rmse:.5f} m, converged={result.converged}')

# Apply alignment to full post cloud
T = result.transformation
post_aligned_xyz = (T[:3, :3] @ post_drifted.xyz.T).T + T[:3, 3]
post_aligned = PointCloud(post_aligned_xyz)

## Point-wise Distance (M3C2-style Signed Elevation Difference)

In [ ]:
# For each pre point, find nearest post point and compute signed dZ
tree_post = KDTree(post_aligned.xyz[:, :2])  # 2D search
dist, idx = tree_post.query(pre.xyz[:, :2], k=1)

# Signed elevation change: post - pre
dz = post_aligned.xyz[idx, 2] - pre.xyz[:, 2]

# Threshold: 0.2 m (2 std dev of sensor noise)
threshold = 0.15
significant_loss = dz < -threshold
significant_gain = dz > threshold
stable = np.abs(dz) <= threshold

print(f'Significant loss (depletion): {significant_loss.sum():,} pts ({significant_loss.mean():.1%})')
print(f'Significant gain (deposition): {significant_gain.sum():,} pts ({significant_gain.mean():.1%})')
print(f'Stable                       : {stable.sum():,} pts ({stable.mean():.1%})')

# Volume proxy (sum of dZ × cell area)
cell_area = (100 / np.sqrt(n)) ** 2  # approximate cell area per point
vol_lost = (-dz[significant_loss]).sum() * cell_area
vol_gained = dz[significant_gain].sum() * cell_area
print(f'\nVolume lost   : {vol_lost:.1f} m³')
print(f'Volume gained : {vol_gained:.1f} m³')
print(f'Net change    : {vol_gained - vol_lost:+.1f} m³')

## Change Map

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Pre DEM
h_pre, xe, ye = np.histogram2d(x, y, bins=80, weights=z_pre)
cnt, _, _ = np.histogram2d(x, y, bins=80)
dem_pre = np.where(cnt > 0, h_pre / cnt, np.nan)
im0 = axes[0].imshow(dem_pre.T, origin='lower', extent=[0, 100, 0, 100], cmap='terrain')
plt.colorbar(im0, ax=axes[0], label='Z (m)'); axes[0].set_title('Pre-Survey DEM')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')

# Post DEM
h_post, _, _ = np.histogram2d(x, y, bins=80, weights=z_post)
dem_post = np.where(cnt > 0, h_post / cnt, np.nan)
im1 = axes[1].imshow(dem_post.T, origin='lower', extent=[0, 100, 0, 100], cmap='terrain')
plt.colorbar(im1, ax=axes[1], label='Z (m)'); axes[1].set_title('Post-Survey DEM')
axes[1].set_xlabel('X (m)')

# Change map
change_map = dem_post - dem_pre
vmax = np.nanpercentile(np.abs(change_map), 98)
im2 = axes[2].imshow(change_map.T, origin='lower', extent=[0, 100, 0, 100],
                     cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(im2, ax=axes[2], label='ΔZ (m)')
axes[2].set_title('Change Map (Post − Pre)')
axes[2].set_xlabel('X (m)')

plt.suptitle('LiDAR Change Detection — Simulated Landslide', fontsize=13)
plt.tight_layout()
plt.savefig('../../outputs/change_detection.png', dpi=150)
plt.show()

**Next**: `11_abovepy_advanced.ipynb`